# Wharton integrity: pinned CPU replay

This notebook is a runnable handoff. **It was not executed on Kaggle in the authoring session.** Import it into an account you control, choose CPU with no accelerator, and enable Internet for the public source checkout if needed. The driver itself makes no network requests.

It replays the closed synthetic development design, with no market data or protected holdout. Reusing these seeds supplies an execution replay, not additional independent evidence. The driver permits one numeric thread and one command at a time, with a 900-second execution budget, 180 seconds per command, and memory admission reserving 1 GiB plus a 512 MiB child allowance. Missing dependencies or resource admission failure stop the run and preserve diagnostics.

The original study completed 288 primary cells, 24 preflight cells and the declared24-cell replay, with no matrix cell failures. All scores were verified. Several apparent gains fail against simpler retained controls; no model is promoted. Full reports and exact source are linked below.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

SOURCE_SHA = "5081e37dadfc9cbb0ae5e06c46385ba89c05e50c"
REPOSITORY = "https://github.com/build-the-future-11/Wharton-Experiment.git"
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
CHECKOUT = WORK / ("wharton_integrity_" + SOURCE_SHA[:12])
if CHECKOUT.exists():
    raise FileExistsError("Use a fresh working directory; existing source is not overwritten: " + str(CHECKOUT))
CHECKOUT.mkdir()
setup_environment = os.environ.copy()
setup_environment["GIT_TERMINAL_PROMPT"] = "0"
setup_deadline = time.monotonic() + 120

def git(*arguments):
    remaining = min(60, int(setup_deadline - time.monotonic()))
    if remaining <= 0:
        raise TimeoutError("Source setup budget exhausted; preserve the partial checkout and start separately")
    result = subprocess.run(["git", "-C", str(CHECKOUT), *arguments],
                            env=setup_environment, check=True, text=True,
                            capture_output=True, timeout=remaining)
    return result.stdout.strip()

git("init", "--quiet")
git("remote", "add", "origin", REPOSITORY)
git("sparse-checkout", "init", "--no-cone")
git("sparse-checkout", "set", "--no-cone", "/src/", "/scripts/", "/tests/", "/protocol/",
    "/pyproject.toml", "/requirements-verified.txt")
git("fetch", "--quiet", "--depth", "1", "--filter=blob:none", "origin", SOURCE_SHA)
git("checkout", "--quiet", "--detach", "FETCH_HEAD")
observed = git("rev-parse", "HEAD")
assert observed == SOURCE_SHA, (SOURCE_SHA, observed)
print("Verified source:", observed)
print("Checkout:", CHECKOUT)
print("Report:", "https://github.com/build-the-future-11/Wharton-Experiment/blob/" + SOURCE_SHA + "/reports/integrity_20261019/RESULTS.md")


## Execute the bounded replay

Source download has its own 120-second setup limit; the scientific driver starts its 900-second budget below. The driver checks installed NumPy, SciPy, scikit-learn, pandas, PyTorch and pytest, and records their actual versions. It does not install packages or assume the authoring host's exact CPU PyTorch wheel exists here. See `ENVIRONMENT_OBSERVED.json` in the pinned repository for the original inventory. Different environments can change floating-point results.

The complete driver plan includes local contract/tensor/gradient tests, two deterministic demo invocations and their comparison, the 288 original development cells in separate world/seed invocations, and raw-score verification after each. Separate seed invocations preserve a checkpoint for each current cell; the original multi-seed study preserved representative first-seed checkpoints. The source is checked for modifications before any model execution.

In [ ]:
from datetime import datetime, timezone

RUN_DIRECTORY = WORK / ("wharton_replay_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
command = [sys.executable, str(CHECKOUT / "scripts/run_integrity_kaggle.py"),
           "--source-sha", SOURCE_SHA, "--output", str(RUN_DIRECTORY), "--matrix"]
print("Starting a new CPU replay:", RUN_DIRECTORY)
completed = subprocess.run(command, cwd=CHECKOUT, check=False)
print("Driver exit status:", completed.returncode)
print("Retained output:", RUN_DIRECTORY)


## Inspect and retrieve

Read the summary below before interpreting any scores. A failed, partial or unstarted result remains a failure or incomplete execution. It must not be presented as a successful 288-cell replay. A `.partial` ZIP is unfinished. The normal `replay_bundle.zip` filename is published only after checksum creation.

Download `replay_bundle.zip` and `replay_bundle.sha256` from this output directory, and save the actual Kaggle notebook/version/run URL with them. The archive includes command receipts, source/data/configuration hashes, environment, predictions, targets, representative cell checkpoints and failures. Verification never needs to unpickle a checkpoint; never load untrusted pickle files. Preserve all results, including those that contradict the model hypothesis.

In [ ]:
import hashlib
import json

summary_path = RUN_DIRECTORY / "execution_summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print(json.dumps({key: summary.get(key) for key in
        ("status", "error", "source_verified", "required_source_sha", "observed_source_sha",
         "matrix_expected", "matrix_completed", "wall_seconds", "child_wall_seconds")}, indent=2))
else:
    print("No terminal summary exists; inspect partial files and treat the run as incomplete.")
archive = RUN_DIRECTORY / "replay_bundle.zip"
checksum = RUN_DIRECTORY / "replay_bundle.sha256"
if archive.is_file() and checksum.is_file():
    actual = hashlib.sha256(archive.read_bytes()).hexdigest()
    expected = checksum.read_text().split()[0]
    assert actual == expected, "Bundle checksum mismatch"
    print("Verified archive SHA256:", actual)
    print("Download:", archive)
    print("Download:", checksum)
else:
    print("No completed archive/checksum pair; retain diagnostics and any .partial file.")
